In [1]:
import warnings
warnings.filterwarnings("ignore")
import random
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [9]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]

In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [12]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [13]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 374)
y_train:  (139,)


(99, 376)

# Feature Selection: RENT

In [14]:
selected_features = ["shape_Sphericity",
                     "glrlm_HighGrayLevelRunEmphasis_PET_c04",
                     "shape_MajorAxisLength",
                     "LBP_102_PET"]

# Selecting features in the DataFrame
X_rent = X[selected_features]

In [15]:
# Selecting features in the DataFrame
X_rent = X[selected_features]
X_new = X_rent.copy()

X_MAASTRO_rent = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_rent.copy()

# Standardization

In [16]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = RobustScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [17]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [18]:
X_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.761164,16.969770,42.073251,0.000000
1,0.697049,15.598394,24.613845,0.000000
2,0.565792,17.334294,48.030294,0.000034
3,0.684364,14.009277,25.589900,0.000000
4,0.503142,21.202180,34.684750,0.000199
...,...,...,...,...
134,0.742102,16.110383,33.069705,0.000000
135,0.722918,21.249575,41.043692,0.000000
136,0.652963,14.873884,36.618802,0.000000
137,0.724255,21.648860,45.870392,0.000000


In [19]:
X_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.704475,0.085656,0.059912,0.000000
1,0.101791,-0.291967,-0.755402,0.000000
2,-1.132016,0.186031,0.338093,0.524263
3,-0.017443,-0.729547,-0.709822,0.000000
4,-1.720923,1.251094,-0.285114,3.028668
...,...,...,...,...
134,0.525285,-0.150985,-0.360532,0.000000
135,0.344965,1.264145,0.011834,0.000000
136,-0.312609,-0.491468,-0.194798,0.000000
137,0.357529,1.374092,0.237230,0.000000


In [20]:
MAASTRO_new

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,0.668072,19.034156,50.002093,0.000026
1,0.669961,11.392444,41.753334,0.000167
2,0.624081,14.567421,44.375483,0.000057
3,0.577624,13.477331,46.115989,0.000000
4,0.630933,16.365554,54.394967,0.000000
...,...,...,...,...
94,0.671754,17.492295,34.218615,0.000000
95,0.632189,14.267900,51.046869,0.000000
96,0.645548,12.143752,50.417953,0.000000
97,0.727488,15.300229,44.901412,0.000000


In [21]:
MAASTRO_new_std

,shape_Sphericity,glrlm_HighGrayLevelRunEmphasis_PET_c04,shape_MajorAxisLength,LBP_102_PET
0,-0.170593,0.654106,0.430171,0.398737
1,-0.152832,-1.450119,0.044973,2.544568
2,-0.584101,-0.575856,0.167421,0.869691
3,-1.020797,-0.876024,0.248699,0.000000
4,-0.519690,-0.080721,0.635308,0.000000
...,...,...,...,...
94,-0.135979,0.229539,-0.306881,0.000000
95,-0.507883,-0.658333,0.478960,0.000000
96,-0.382318,-1.243239,0.449591,0.000000
97,0.387916,-0.374070,0.191981,0.000000


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [22]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 16:31:18,866] A new study created in memory with name: no-name-d05ffb63-2b6c-4117-819d-f63d35554a0e


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.7452471482889734


[I 2024-04-14 16:31:27,721] A new study created in memory with name: no-name-1927faca-d25a-441a-902e-7dab84fb4ccb


Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:31:27,707] Trial 0 finished with value: 0.6885812252956434 and parameters: {}. Best is trial 0 with value: 0.6885812252956434.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6885812252956434], datetime_start=datetime.datetime(2024, 4, 14, 16, 31, 18, 988674), datetime_complete=datetime.datetime(2024, 4, 14, 16, 31, 27, 706442), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6885812252956434


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24445113125013962
Fold 2 IBS: 0.19124878012978022
Fold 3 IBS: 0.20665297767216081
Fold 4 IBS: 0.19438765965098836
Fold 5 IBS: 0.19775948988265402
[I 2024-04-14 16:31:28,489] Trial 0 finished with value: 0.20690000771714462 and parameters: {}. Best is trial 0 with value: 0.20690000771714462.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.20690000771714462], datetime_start=datetime.datetime(2024, 4, 14, 16, 31, 27, 878614), datetime_complete=datetime.datetime(2024, 4, 14, 16, 31, 28, 489067), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.20690000771714462


In [23]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [24]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.689
train_ibs:  0.207


#### Test

In [25]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [26]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.532
IBS score: 0.29


In [27]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [28]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [29]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 16:31:28,850] A new study created in memory with name: no-name-430ba2bd-467d-4ec4-8316-1aeb7c9de30b


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.599601593625498


[I 2024-04-14 16:31:29,122] A new study created in memory with name: no-name-d7047135-8ed6-4c89-8aa8-e9b99fefcba6


Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.5595744680851064
Fold 4 C-index: 0.7072243346007605
Fold 5 C-index: 0.6952789699570815
[I 2024-04-14 16:31:29,103] Trial 0 finished with value: 0.6603978887575652 and parameters: {}. Best is trial 0 with value: 0.6603978887575652.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6603978887575652], datetime_start=datetime.datetime(2024, 4, 14, 16, 31, 28, 889267), datetime_complete=datetime.datetime(2024, 4, 14, 16, 31, 29, 103202), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6603978887575652


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709805071834
Fold 2 IBS: 0.2320398756992525
Fold 3 IBS: 0.2289818676351831
Fold 4 IBS: 0.2419747644101402
Fold 5 IBS: 0.22939558596182483
[I 2024-04-14 16:31:29,532] Trial 0 finished with value: 0.23592783835142378 and parameters: {}. Best is trial 0 with value: 0.23592783835142378.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592783835142378], datetime_start=datetime.datetime(2024, 4, 14, 16, 31, 29, 203925), datetime_complete=datetime.datetime(2024, 4, 14, 16, 31, 29, 532463), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592783835142378


In [30]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [31]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.66
train_ibs:  0.236


#### Test

In [32]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [33]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.535


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [34]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [35]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 16:31:29,752] A new study created in memory with name: no-name-ca02c333-dbd5-4161-805b-26a8fcd5d45a


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872


[I 2024-04-14 16:31:30,317] A new study created in memory with name: no-name-15dd2cad-4d63-4ff8-b0d0-72281b86ab6b


Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:31:30,288] Trial 0 finished with value: 0.6910259146234387 and parameters: {}. Best is trial 0 with value: 0.6910259146234387.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6910259146234387], datetime_start=datetime.datetime(2024, 4, 14, 16, 31, 29, 804985), datetime_complete=datetime.datetime(2024, 4, 14, 16, 31, 30, 287892), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6910259146234387


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24385936084363596
Fold 2 IBS: 0.19104145101471223
Fold 3 IBS: 0.2076394199174915
Fold 4 IBS: 0.19441380426749091
Fold 5 IBS: 0.1981557680324636
[I 2024-04-14 16:31:30,840] Trial 0 finished with value: 0.20702196081515883 and parameters: {}. Best is trial 0 with value: 0.20702196081515883.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.20702196081515883], datetime_start=datetime.datetime(2024, 4, 14, 16, 31, 30, 358620), datetime_complete=datetime.datetime(2024, 4, 14, 16, 31, 30, 840449), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.20702196081515883


In [36]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [37]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.691
train_ibs:  0.207


#### Test 

In [38]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [39]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.288


In [40]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [41]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 16:31:31,262] A new study created in memory with name: no-name-54f73c05-a366-4012-ab35-2873daae3b29


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:31:31,703] Trial 0 finished with value: 0.6901748507936514 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6901748507936514.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6824034334763949
[I 2024-04-14 16:31:32,136] Trial 1 finished with value: 0.6885196689459349 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.6901748507936514.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:31:32,520] Trial 2 finished with value: 0.6893780380446474 and parameters: {'l1_ratio': 0.22692876841884668}. Be

Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:31:45,195] Trial 24 finished with value: 0.6901748507936514 and parameters: {'l1_ratio': 0.7805647035680947}. Best is trial 6 with value: 0.6910259146234387.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:31:46,002] Trial 25 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.9368557715647121}. Best is trial 6 with value: 0.6910259146234387.
Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6996197718631179
Fold 5 C-index: 0.703862660944206
[I 2024-04-14 16:31:46,427] Trial 26 finished with value: 0.6567448709556052 and parameters: {'l1_ratio': 0.015423757551295547}. Best is trial 6 with value: 0.691

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:32:00,748] Trial 48 finished with value: 0.6901748507936514 and parameters: {'l1_ratio': 0.8577244154633872}. Best is trial 6 with value: 0.6910259146234387.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6808510638297872
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:32:01,293] Trial 49 finished with value: 0.6910259146234387 and parameters: {'l1_ratio': 0.9586049921503356}. Best is trial 6 with value: 0.6910259146234387.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.676595744680851
Fold 4 C-index: 0.7452471482889734
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:32:01,753] Trial 50 finished with value: 0.6901748507936514 and parameters: {'l1_ratio': 0.7809764372372684}. 

...

Fold 1 IBS: 0.24384372425403145
Fold 2 IBS: 0.19102097719113648
Fold 3 IBS: 0.20732370743921147
Fold 4 IBS: 0.1943604220569732
Fold 5 IBS: 0.19804380439121627
[I 2024-04-14 16:32:27,415] Trial 0 finished with value: 0.20691852706651379 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.20691852706651379.
Fold 1 IBS: 0.24385693771373115
Fold 2 IBS: 0.19096890975333564
Fold 3 IBS: 0.20666536420831402
Fold 4 IBS: 0.19423126307775032
Fold 5 IBS: 0.1978098990568525
[I 2024-04-14 16:32:28,120] Trial 1 finished with value: 0.20670647476199672 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.20670647476199672.
Fold 1 IBS: 0.24385714315300536
Fold 2 IBS: 0.1909610640349487
Fold 3 IBS: 0.20653138473563162
Fold 4 IBS: 0.19420655067123255
Fold 5 IBS: 0.1977610598434761
[I 2024-04-14 16:32:28,783] Trial 2 finished with value: 0.20666344048765883 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.2066634404876

Fold 1 IBS: 0.24389125940108533
Fold 2 IBS: 0.1909321892340936
Fold 3 IBS: 0.2287320438949615
Fold 4 IBS: 0.19411853822138028
Fold 5 IBS: 0.19764125279155897
[I 2024-04-14 16:32:42,302] Trial 25 finished with value: 0.21106305670861594 and parameters: {'l1_ratio': 0.10297700483781348}. Best is trial 20 with value: 0.20655330517822393.
Fold 1 IBS: 0.24384175324356847
Fold 2 IBS: 0.19097041920242336
Fold 3 IBS: 0.20680750845909757
Fold 4 IBS: 0.19425414285743003
Fold 5 IBS: 0.19786339608895115
[I 2024-04-14 16:32:43,041] Trial 26 finished with value: 0.2067474439702941 and parameters: {'l1_ratio': 0.3527001258697712}. Best is trial 20 with value: 0.20655330517822393.
Fold 1 IBS: 0.24383187824404834
Fold 2 IBS: 0.19100929740110129
Fold 3 IBS: 0.20718148857936933
Fold 4 IBS: 0.19433225538465687
Fold 5 IBS: 0.1979936739277619
[I 2024-04-14 16:32:43,940] Trial 27 finished with value: 0.20686971870738752 and parameters: {'l1_ratio': 0.58416616744615}. Best is trial 20 with value: 0.2065533051

Fold 5 IBS: 0.19789859412874944
[I 2024-04-14 16:32:59,120] Trial 49 finished with value: 0.20677974046363135 and parameters: {'l1_ratio': 0.402705511740666}. Best is trial 20 with value: 0.20655330517822393.
Fold 1 IBS: 0.24384018438088706
Fold 2 IBS: 0.19094548132286418
Fold 3 IBS: 0.20639786985872818
Fold 4 IBS: 0.1941778033859872
Fold 5 IBS: 0.19771450326274714
[I 2024-04-14 16:32:59,673] Trial 50 finished with value: 0.20661516844224276 and parameters: {'l1_ratio': 0.17526057356525065}. Best is trial 20 with value: 0.20655330517822393.
Fold 1 IBS: 0.24385251050406204
Fold 2 IBS: 0.1909412262689443
Fold 3 IBS: 0.20624750352205673
Fold 4 IBS: 0.1941322816934018
Fold 5 IBS: 0.1976578888489821
[I 2024-04-14 16:33:00,489] Trial 51 finished with value: 0.2065662821674894 and parameters: {'l1_ratio': 0.11717503156407644}. Best is trial 20 with value: 0.20655330517822393.
Fold 1 IBS: 0.24386226617594486
Fold 2 IBS: 0.2259746889406444
Fold 3 IBS: 0.2287577295619982
Fold 4 IBS: 0.2372338088

Fold 2 IBS: 0.1909207215842509
Fold 3 IBS: 0.22874418247192937
Fold 4 IBS: 0.2367145075877851
Fold 5 IBS: 0.1976328176965277
[I 2024-04-14 16:33:16,514] Trial 74 finished with value: 0.2195773325072455 and parameters: {'l1_ratio': 0.0898435928340312}. Best is trial 20 with value: 0.20655330517822393.
Fold 1 IBS: 0.24385792264679257
Fold 2 IBS: 0.19094892430005037
Fold 3 IBS: 0.20633026634608104
Fold 4 IBS: 0.19415063727124496
Fold 5 IBS: 0.1976874814924129
[I 2024-04-14 16:33:17,417] Trial 75 finished with value: 0.20659504641131637 and parameters: {'l1_ratio': 0.14629098567939705}. Best is trial 20 with value: 0.20655330517822393.
Fold 1 IBS: 0.24388972547886142
Fold 2 IBS: 0.1909334298052968
Fold 3 IBS: 0.2062276175202841
Fold 4 IBS: 0.19414542658059258
Fold 5 IBS: 0.19765260292402545
[I 2024-04-14 16:33:18,273] Trial 76 finished with value: 0.2065697604618121 and parameters: {'l1_ratio': 0.11383137001752158}. Best is trial 20 with value: 0.20655330517822393.
Fold 1 IBS: 0.2438674650

Fold 5 IBS: 0.1976561470934536
[I 2024-04-14 16:33:33,813] Trial 98 finished with value: 0.20656260980263372 and parameters: {'l1_ratio': 0.11605601047082038}. Best is trial 20 with value: 0.20655330517822393.
Fold 1 IBS: 0.24387273010897204
Fold 2 IBS: 0.1909376189870331
Fold 3 IBS: 0.20644157937053642
Fold 4 IBS: 0.19417714363141486
Fold 5 IBS: 0.1977263150373932
[I 2024-04-14 16:33:34,394] Trial 99 finished with value: 0.20663107742706993 and parameters: {'l1_ratio': 0.1874631498967019}. Best is trial 20 with value: 0.20655330517822393.


* Best trial for IBS: 
 FrozenTrial(number=20, state=TrialState.COMPLETE, values=[0.20655330517822393], datetime_start=datetime.datetime(2024, 4, 14, 16, 32, 38, 766426), datetime_complete=datetime.datetime(2024, 4, 14, 16, 32, 39, 439626), params={'l1_ratio': 0.10500472511016506}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=20, value=Non

In [42]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [43]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.691
train_ibs:  0.207


#### Test

In [44]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.980766121964777)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.10500472511016506)

test_ibs:  0.288


In [46]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [47]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 16:33:34,843] A new study created in memory with name: no-name-614f556a-a885-4a70-a632-b28a16876311


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6454183266932271
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.6723404255319149
Fold 4 C-index: 0.7281368821292775
Fold 5 C-index: 0.648068669527897
[I 2024-04-14 16:33:41,705] Trial 0 finished with value: 0.6977075894586338 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6977075894586338.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.7276595744680852
Fold 4 C-index: 0.7338403041825095
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:33:48,416] Trial 1 finished with value: 0.7128781680014569 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 3 C-index: 0.774468085106383
Fold 4 C-index: 0.7813688212927756
Fold 5 C-index: 0.703862660944206
[I 2024-04-14 16:34:45,708] Trial 15 finished with value: 0.7359974816455777 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 6, 'min_samples_leaf': 16, 'max_depth': 6, 'n_estimators': 135, 'oob_score': True, 'max_samples': 0.9916077098875709, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2050785814281292, 'warm_start': True}. Best is trial 14 with value: 0.7401429490839831.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.8178294573643411
Fold 3 C-index: 0.7914893617021277
Fold 4 C-index: 0.779467680608365
Fold 5 C-index: 0.7060085836909872
[I 2024-04-14 16:34:46,010] Trial 16 finished with value: 0.7424649927687818 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 7, 'min_samples_leaf': 16, 'max_depth': 7, 'n_estimators': 7, 'oob_score': True, 'max_samples': 0.9632036286697128, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19855793167729388, 

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.813953488372093
Fold 3 C-index: 0.8468085106382979
Fold 4 C-index: 0.8174904942965779
Fold 5 C-index: 0.7725321888412017
[I 2024-04-14 16:35:10,931] Trial 30 finished with value: 0.7736629125252518 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 17, 'n_estimators': 369, 'oob_score': True, 'max_samples': 0.6833587310905035, 'max_features': None, 'min_weight_fraction_leaf': 0.008532800179007469, 'warm_start': True}. Best is trial 29 with value: 0.7751733627229969.
Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.8178294573643411
Fold 3 C-index: 0.8425531914893617
Fold 4 C-index: 0.8136882129277566
Fold 5 C-index: 0.7639484978540773
[I 2024-04-14 16:35:14,319] Trial 31 finished with value: 0.7735002862697369 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 20, 'n_estimators': 379, 'oob_score': True, 'max_samples': 0.6547593902381754,

...

Fold 1 IBS: 0.22612682993385486
Fold 2 IBS: 0.17223298934748948
Fold 3 IBS: 0.20782358499736456
Fold 4 IBS: 0.20181112716562463
Fold 5 IBS: 0.21350833459590543
[I 2024-04-14 16:41:18,110] Trial 32 finished with value: 0.2043005732080478 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 13, 'max_depth': 8, 'n_estimators': 316, 'oob_score': False, 'max_samples': 0.8753375341925101, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.2230974552568295}. Best is trial 1 with value: 0.20384699429436565.
Fold 1 IBS: 0.22177687649658975
Fold 2 IBS: 0.17630377861515312
Fold 3 IBS: 0.2122309287307206
Fold 4 IBS: 0.2017574245411477
Fold 5 IBS: 0.21470545803951144
[I 2024-04-14 16:41:22,739] Trial 33 finished with value: 0.20535489328462447 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 307, 'oob_score': False, 'max_samples': 0.8780403780133842, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.23790435513187447
Fold 2 IBS: 0.16917310562036944
Fold 3 IBS: 0.2032724536335342
Fold 4 IBS: 0.20275263571784216
Fold 5 IBS: 0.21198800508925086
[I 2024-04-14 16:42:10,738] Trial 48 finished with value: 0.20501811103857426 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 14, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 369, 'oob_score': False, 'max_samples': 0.862646612475437, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.18882619674434536}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.21643739431200149
Fold 2 IBS: 0.18002210069568697
Fold 3 IBS: 0.20963559196115233
Fold 4 IBS: 0.20201711337877098
Fold 5 IBS: 0.21711926927169345
[I 2024-04-14 16:42:12,111] Trial 49 finished with value: 0.20504629392386103 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 12, 'min_samples_leaf': 19, 'max_depth': 7, 'n_estimators': 68, 'oob_score': False, 'max_samples': 0.712240935894755, 'max_features': 'sqrt', 'min_weight_fraction

Fold 5 IBS: 0.21281753768850425
[I 2024-04-14 16:42:42,772] Trial 63 finished with value: 0.20478734721351177 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 16, 'min_samples_leaf': 12, 'max_depth': 6, 'n_estimators': 145, 'oob_score': False, 'max_samples': 0.5771231216606082, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.05763095148942208}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.21904071817956408
Fold 2 IBS: 0.17322770907939164
Fold 3 IBS: 0.21071337338690455
Fold 4 IBS: 0.20198953222034577
Fold 5 IBS: 0.21531568867352077
[I 2024-04-14 16:42:43,804] Trial 64 finished with value: 0.20405740430794536 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 2, 'n_estimators': 74, 'oob_score': False, 'max_samples': 0.6331344922813995, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1612709153840626}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.2211019166635484
Fold 2 IBS: 0.175

Fold 1 IBS: 0.22242518635376562
Fold 2 IBS: 0.17356537149199874
Fold 3 IBS: 0.19348965127184473
Fold 4 IBS: 0.20729301862706373
Fold 5 IBS: 0.21518309439089522
[I 2024-04-14 16:43:04,333] Trial 79 finished with value: 0.2023912644271136 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 6, 'n_estimators': 25, 'oob_score': False, 'max_samples': 0.7940170614934956, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19696931079449848}. Best is trial 40 with value: 0.2022791344019581.
Fold 1 IBS: 0.2449767955817733
Fold 2 IBS: 0.172095460075217
Fold 3 IBS: 0.21463433607213933
Fold 4 IBS: 0.21510140082957746
Fold 5 IBS: 0.21527266884222518
[I 2024-04-14 16:43:05,188] Trial 80 finished with value: 0.21241613228018644 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 10, 'max_depth': 6, 'n_estimators': 28, 'oob_score': False, 'max_samples': 0.8170871700419534, 'max_features': None, 'min_weight_fraction_leaf

Fold 5 IBS: 0.21635495382740025
[I 2024-04-14 16:43:18,507] Trial 94 finished with value: 0.2055660070329798 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 12, 'min_samples_leaf': 16, 'max_depth': 13, 'n_estimators': 93, 'oob_score': False, 'max_samples': 0.713903532939743, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.22122568041634558}. Best is trial 86 with value: 0.20204409612264693.
Fold 1 IBS: 0.26932513184118945
Fold 2 IBS: 0.1697323509215535
Fold 3 IBS: 0.20905255797983618
Fold 4 IBS: 0.2258672184082545
Fold 5 IBS: 0.2152407835992728
[I 2024-04-14 16:43:19,030] Trial 95 finished with value: 0.21784360855002127 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 11, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 25, 'oob_score': False, 'max_samples': 0.8547756485942715, 'max_features': None, 'min_weight_fraction_leaf': 0.17594983744388534}. Best is trial 86 with value: 0.20204409612264693.
Fold 1 IBS: 0.22065985212578162
Fold 2 IBS: 0.172404

In [48]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [49]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.83
train_ibs:  0.202


#### Test

In [50]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [51]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=18, max_leaf_nodes=11,
                     max_samples=0.9059465751594485, min_samples_leaf=1,
                     min_weight_fraction_leaf=0.029068528038335394,
                     n_estimators=318, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.544


RandomSurvivalForest(max_depth=13, max_features='log2', max_leaf_nodes=11,
                     max_samples=0.7105780352803006, min_samples_leaf=11,
                     min_samples_split=13,
                     min_weight_fraction_leaf=0.2021626029496649,
                     n_estimators=31, random_state=123)

test_ibs:  0.247


In [52]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [53]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 16:43:25,568] A new study created in memory with name: no-name-45748c55-979d-4b1d-a3be-15efd46f1f6d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.8042553191489362
Fold 4 C-index: 0.7490494296577946
Fold 5 C-index: 0.6909871244635193
[I 2024-04-14 16:43:26,735] Trial 0 finished with value: 0.7329375617815092 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7329375617815092.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 16:43:29,242] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 16:43:56,004] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 20, 'n_estimators': 246, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.12362405012814498, 'min_weight_fraction_leaf': 0.18949444225079395}. Best is trial 8 with value: 0.7362139149697521.
Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.7906976744186046
Fold 3 C-index: 0.7872340425531915
Fold 4 C-index: 0.7604562737642585
Fold 5 C-index: 0.6909871244635193
[I 2024-04-14 16:43:57,139] Trial 17 finished with value: 0.7333650628805521 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 9, 'n_estimators': 362, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8640754037508087, 'min_weight_fraction_leaf': 0.08300984976320

Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.7914893617021277
Fold 4 C-index: 0.7566539923954373
Fold 5 C-index: 0.6866952789699571
[I 2024-04-14 16:44:24,254] Trial 31 finished with value: 0.7301852441402965 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 9, 'min_samples_leaf': 10, 'max_depth': 11, 'n_estimators': 288, 'oob_score': False, 'warm_start': True, 'max_features': 0.1, 'max_samples': 0.8780867392507613, 'min_weight_fraction_leaf': 0.08053342835298444}. Best is trial 8 with value: 0.7362139149697521.
Fold 1 C-index: 0.6254980079681275
Fold 2 C-index: 0.7868217054263565
Fold 3 C-index: 0.7829787234042553
Fold 4 C-index: 0.7490494296577946
Fold 5 C-index: 0.6909871244635193
[I 2024-04-14 16:44:25,558] Trial 32 finished with value: 0.7270669981840107 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.2369623605519169
Fold 2 IBS: 0.2015664827514855
Fold 3 IBS: 0.2044196732107329
Fold 4 IBS: 0.2152719150958778
Fold 5 IBS: 0.2123358137500061
[I 2024-04-14 16:49:20,727] Trial 30 finished with value: 0.21411124907200385 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.20923338228172916.
Fold 1 IBS: 0.23451695205118067
Fold 2 IBS: 0.19751010562426458
Fold 3 IBS: 0.2027198084019687
Fold 4 IBS: 0.21411412044401185
Fold 5 IBS: 0.2111761169753505
[I 2024-04-14 16:49:26,146] Trial 31 finished with value: 0.21200742069935527 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 8, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.923

Fold 1 IBS: 0.23247724033039832
Fold 2 IBS: 0.18748554961121114
Fold 3 IBS: 0.2090170455982855
Fold 4 IBS: 0.207742375469341
Fold 5 IBS: 0.20889347688505747
[I 2024-04-14 16:51:06,212] Trial 45 finished with value: 0.2091231375788587 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 14, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 144, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8505483502716148, 'min_weight_fraction_leaf': 0.09794633817349793}. Best is trial 45 with value: 0.2091231375788587.
Fold 1 IBS: 0.2314286160573986
Fold 2 IBS: 0.1860889745854672
Fold 3 IBS: 0.20638267011548084
Fold 4 IBS: 0.20884587453459477
Fold 5 IBS: 0.20805701088484838
[I 2024-04-14 16:51:09,499] Trial 46 finished with value: 0.20816062923555795 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 138, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.77092

Fold 1 IBS: 0.2376464824369693
Fold 2 IBS: 0.21593994734134395
Fold 3 IBS: 0.22177987321764975
Fold 4 IBS: 0.22790863488656224
Fold 5 IBS: 0.2205529164521157
[I 2024-04-14 16:51:53,466] Trial 60 finished with value: 0.22476557086692814 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 74, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.6041879821929975, 'min_weight_fraction_leaf': 0.1056236625824671}. Best is trial 46 with value: 0.20816062923555795.
Fold 1 IBS: 0.23781793031064252
Fold 2 IBS: 0.1862495208476139
Fold 3 IBS: 0.20248456745784213
Fold 4 IBS: 0.21234711020159586
Fold 5 IBS: 0.2076972639565933
[I 2024-04-14 16:51:57,028] Trial 61 finished with value: 0.20931927855485752 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 19, 'n_estimators': 197, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.71011260

Fold 1 IBS: 0.23693321208405524
Fold 2 IBS: 0.19000973925047765
Fold 3 IBS: 0.20243881773088923
Fold 4 IBS: 0.2110667431713562
Fold 5 IBS: 0.20765848231427028
[I 2024-04-14 16:52:23,679] Trial 75 finished with value: 0.20962139891020973 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 18, 'n_estimators': 104, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.49599352625156035, 'min_weight_fraction_leaf': 0.03138093454428739}. Best is trial 71 with value: 0.20716724376288945.
Fold 1 IBS: 0.2375162774090104
Fold 2 IBS: 0.21696007860868566
Fold 3 IBS: 0.2199964189465905
Fold 4 IBS: 0.2298197447784768
Fold 5 IBS: 0.2208032877666601
[I 2024-04-14 16:52:25,349] Trial 76 finished with value: 0.2250191615018847 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 14, 'min_samples_leaf': 1, 'max_depth': 17, 'n_estimators': 81, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.6498539

Fold 1 IBS: 0.23900229487413618
Fold 2 IBS: 0.1829989350900894
Fold 3 IBS: 0.20504008460974205
Fold 4 IBS: 0.21135329415618298
Fold 5 IBS: 0.20658610615550185
[I 2024-04-14 16:53:10,085] Trial 90 finished with value: 0.20899614297713048 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 1, 'max_depth': 16, 'n_estimators': 95, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7812194815989179, 'min_weight_fraction_leaf': 0.052583924158953604}. Best is trial 71 with value: 0.20716724376288945.
Fold 1 IBS: 0.23336269917153504
Fold 2 IBS: 0.18793910685243745
Fold 3 IBS: 0.20227191981491185
Fold 4 IBS: 0.2126322523584538
Fold 5 IBS: 0.20891055719221097
[I 2024-04-14 16:53:13,423] Trial 91 finished with value: 0.2090233070779098 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 170, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7

In [54]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [55]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.764
train_ibs:  0.207


#### Test

In [56]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [57]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=6, max_features=None, max_leaf_nodes=13,
                   max_samples=0.9529576562335005, min_samples_split=15,
                   min_weight_fraction_leaf=0.05156899271012762,
                   n_estimators=376, random_state=123, warm_start=True)

C-index score: 0.543


ExtraSurvivalTrees(max_depth=19, max_features=None, max_leaf_nodes=17,
                   max_samples=0.6638490929451707, min_samples_leaf=2,
                   min_samples_split=19,
                   min_weight_fraction_leaf=0.052914450220883674,
                   n_estimators=60, oob_score=True, random_state=123)

IBS: 0.256


In [58]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [59]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 16:53:33,341] A new study created in memory with name: no-name-0a463761-879b-4ec8-a00c-1b344565fff0


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 16:53:57,774] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 16:54:10,739] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:02:04,580] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7079199845805899.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:02:56,394] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:12:23,128] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.7079199845805899.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:12:42,951] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 18:34:45,656] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 53 with value: 0.2348064399929884.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 18:35:39,301] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.357604

Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 18:42:49,553] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8952643616873973, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.6509686174553241, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 'auto', 'min_impurity_decrease': 2.2946753237767036e-06, 'validation_fraction': 0.8670144195682054, 'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 18}. Best is trial 64 with value: 0.23475649727610257.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 18:43:12,554] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9559398584951578, 'learning_rate': 0.001312025546225645, 'dropout_rate': 0.8046

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 18:51:18,029] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9040601608260395, 'learning_rate': 0.008028762030775153, 'dropout_rate': 0.1661055390191164, 'n_estimators': 412, 'criterion': 'squared_error', 'ccp_alpha': 0.5920167940309409, 'min_weight_fraction_leaf': 0.20760648939509768, 'max_features': 1, 'min_impurity_decrease': 1.2858411523836384e-06, 'validation_fraction': 0.7519570151291436, 'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 64 with value: 0.23475649727610257.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 18:51:49,050] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.003888190949209832, 'dropout_rate': 0.625165508

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 19:01:42,170] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7762725560789899, 'learning_rate': 0.0030671608519508686, 'dropout_rate': 0.2500967923284591, 'n_estimators': 479, 'criterion': 'squared_error', 'ccp_alpha': 1.2742275973203492, 'min_weight_fraction_leaf': 0.27772784044341225, 'max_features': 'auto', 'min_impurity_decrease': 0.0011701047770450368, 'validation_fraction': 0.9552938036430088, 'min_samples_split': 13, 'max_leaf_nodes': 19, 'min_samples_leaf': 17, 'max_depth': 2}. Best is trial 85 with value: 0.23301282639832266.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 19:03:03,332] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9240076064991638, 'learning_rate': 0.036372907201929795, 'dropout_rate': 0.166

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 19:18:47,107] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9084360842440081, 'learning_rate': 0.06808689488183181, 'dropout_rate': 0.25443633448028735, 'n_estimators': 437, 'criterion': 'squared_error', 'ccp_alpha': 0.5386043790221791, 'min_weight_fraction_leaf': 0.03403879613376609, 'max_features': 'auto', 'min_impurity_decrease': 8.69596280226011e-07, 'validation_fraction': 0.9221483446288596, 'min_samples_split': 20, 'max_leaf_nodes': 12, 'min_samples_leaf': 16, 'max_depth': 2}. Best is trial 85 with value: 0.23301282639832266.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.23301282639832266], datetime_start=datetime.datetime(2024, 4, 14, 18, 57, 22, 418559), datetime_complete=datetime.datetime(2024, 4, 14, 18, 58, 36, 467993), params={'subsample': 0.9481727373988897, 'learning_rate': 0.0202297525909

In [60]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [61]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.713
train_ibs:  0.233


#### Test

In [62]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [63]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.022737959076050005,
                                 dropout_rate=0.8285090704895912,
                                 learning_rate=0.0513322635401625, max_depth=2,
                                 max_features='sqrt', max_leaf_nodes=10,
                                 min_impurity_decrease=0.002070496973827786,
                                 min_samples_leaf=15, min_samples_split=19,
                                 min_weight_fraction_leaf=0.3101145382804477,
                                 n_estimators=485, random_state=123,
                                 subsample=0.45214538811231725,
                                 validation_fraction=0.9071319756271838)

C-index score: 0.532


GradientBoostingSurvivalAnalysis(ccp_alpha=0.01138981446692314,
                                 criterion='squared_error',
                                 dropout_rate=0.2479619862207481,
                                 learning_rate=0.02022975259096714, max_depth=8,
                                 max_features='auto', max_leaf_nodes=20,
                                 min_impurity_decrease=9.61920586779085e-07,
                                 min_samples_leaf=18, min_samples_split=14,
                                 min_weight_fraction_leaf=0.026767353450782638,
                                 n_estimators=438, random_state=123,
                                 subsample=0.9481727373988897,
                                 validation_fraction=0.973754058089352)

IBS: 0.229


In [64]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [65]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 19:19:15,510] A new study created in memory with name: no-name-51142c8f-fd7a-45f6-96dd-cab1d3a40f2d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-14 19:19:16,853] Trial 0 finished with value: 0.6232514766080982 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6232514766080982.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6523605150214592
[I 2024-04-14 19:19:28,949] Trial 1 finished with value: 0.6232514766080982 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6232514766080982.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.7403100775193798
Fold 3 C-index: 0.5872340425531914
Fold 4 C-index: 0.6844106463878327
Fold 5 C-index: 0.6824034334763949
[I 2024-04-14 19:21:25,082] Trial 19 finished with value: 0.6560031140909454 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 12 with value: 0.657849549557435.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6781115879828327
[I 2024-04-14 19:21:39,847] Trial 20 finished with value: 0.6424033495535489 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 12 with value: 0.657849549557435.
Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6042553191489362
F

Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6695278969957081
[I 2024-04-14 19:22:47,531] Trial 38 finished with value: 0.6343121093640922 and parameters: {'subsample': 0.24816424105467477, 'dropout_rate': 0.3720991365773877, 'n_estimators': 35, 'learning_rate': 0.024453949190947188}. Best is trial 27 with value: 0.6672323922890245.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6501901140684411
Fold 5 C-index: 0.6738197424892703
[I 2024-04-14 19:22:52,434] Trial 39 finished with value: 0.634540710565314 and parameters: {'subsample': 0.3071617415896747, 'dropout_rate': 0.4352656711347185, 'n_estimators': 287, 'learning_rate': 0.0757363212737669}. Best is trial 27 with value: 0.6672323922890245.
Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6425855513307985
Fo

...

Fold 3 IBS: 0.27495840642227154
Fold 4 IBS: 0.2403770465027076
Fold 5 IBS: 0.22389919673997055
[I 2024-04-14 19:27:53,162] Trial 19 finished with value: 0.24211794449286744 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.24532728078212304
Fold 2 IBS: 0.21338053568843193
Fold 3 IBS: 0.23280583204014674
Fold 4 IBS: 0.23050963160208696
Fold 5 IBS: 0.21229213294998353
[I 2024-04-14 19:27:54,323] Trial 20 finished with value: 0.22686308261255445 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.24480948267779246
Fold 2 IBS: 0.217894217650721
Fold 3 IBS: 0.23043919128917217
Fold 4 IBS: 0.23275995527788665
Fold 5 IBS: 0.215372968667489
[I 2024-04-14 19:27:55,171] Trial 21 finis

Fold 4 IBS: 0.23376225132374828
Fold 5 IBS: 0.21475575962300342
[I 2024-04-14 19:28:30,584] Trial 38 finished with value: 0.23328897274995045 and parameters: {'subsample': 0.605229739689187, 'dropout_rate': 0.13272164755980653, 'n_estimators': 176, 'learning_rate': 0.01283698591663533}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.29108106719697524
Fold 2 IBS: 0.21415810554809628
Fold 3 IBS: 0.2917820507930866
Fold 4 IBS: 0.2588982634047282
Fold 5 IBS: 0.24407574900323586
[I 2024-04-14 19:28:31,375] Trial 39 finished with value: 0.25999904718922445 and parameters: {'subsample': 0.6970162917669863, 'dropout_rate': 0.48841341315505027, 'n_estimators': 59, 'learning_rate': 0.06892741183938003}. Best is trial 17 with value: 0.2265683943607694.
Fold 1 IBS: 0.2925659515921918
Fold 2 IBS: 0.2140746731129848
Fold 3 IBS: 0.2919432513336724
Fold 4 IBS: 0.257477121073266
Fold 5 IBS: 0.24657769529804316
[I 2024-04-14 19:28:32,508] Trial 40 finished with value: 0.2605277384820316 

Fold 5 IBS: 0.27450442032616973
[I 2024-04-14 19:28:49,814] Trial 57 finished with value: 0.29085967593170914 and parameters: {'subsample': 0.763955514973389, 'dropout_rate': 0.9964559295946985, 'n_estimators': 282, 'learning_rate': 0.02203371126489022}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.2514876544143989
Fold 2 IBS: 0.20473165445913052
Fold 3 IBS: 0.2442786298594191
Fold 4 IBS: 0.2292932477615486
Fold 5 IBS: 0.2122783863810015
[I 2024-04-14 19:28:50,406] Trial 58 finished with value: 0.2284139145750997 and parameters: {'subsample': 0.8210665501105633, 'dropout_rate': 0.9557982366790785, 'n_estimators': 50, 'learning_rate': 0.03153701184110286}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.2450009034050451
Fold 2 IBS: 0.21574021216390366
Fold 3 IBS: 0.2316126494096215
Fold 4 IBS: 0.2313085066143514
Fold 5 IBS: 0.2135215585555951
[I 2024-04-14 19:28:50,762] Trial 59 finished with value: 0.22743676602970336 and parameters: {'subsample': 0.7

Fold 5 IBS: 0.21457551785343906
[I 2024-04-14 19:29:23,079] Trial 76 finished with value: 0.2278190214501346 and parameters: {'subsample': 0.5683268745859826, 'dropout_rate': 0.8653081987830238, 'n_estimators': 34, 'learning_rate': 0.020347121996093395}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.24975605957417094
Fold 2 IBS: 0.20544669926019324
Fold 3 IBS: 0.24292589881032364
Fold 4 IBS: 0.2283240329592914
Fold 5 IBS: 0.21054396005104126
[I 2024-04-14 19:29:24,641] Trial 77 finished with value: 0.22739933013100408 and parameters: {'subsample': 0.5096986308003991, 'dropout_rate': 0.49881327862855496, 'n_estimators': 145, 'learning_rate': 0.010267257888313383}. Best is trial 56 with value: 0.22653496264114223.
Fold 1 IBS: 0.24684927097070503
Fold 2 IBS: 0.2094952280230178
Fold 3 IBS: 0.23652719105524916
Fold 4 IBS: 0.22913951054643386
Fold 5 IBS: 0.2106573202990221
[I 2024-04-14 19:29:25,332] Trial 78 finished with value: 0.2265337041788856 and parameters: {'subsamp

Fold 5 IBS: 0.21018857248815617
[I 2024-04-14 19:29:46,645] Trial 95 finished with value: 0.2259993215506233 and parameters: {'subsample': 0.37798247189331685, 'dropout_rate': 0.8572208063796695, 'n_estimators': 112, 'learning_rate': 0.01063058308878333}. Best is trial 95 with value: 0.2259993215506233.
Fold 1 IBS: 0.2471425193657016
Fold 2 IBS: 0.20595699893541525
Fold 3 IBS: 0.2415748835282002
Fold 4 IBS: 0.22741639083301954
Fold 5 IBS: 0.209543331672624
[I 2024-04-14 19:29:48,632] Trial 96 finished with value: 0.22632682486699213 and parameters: {'subsample': 0.34523349906195283, 'dropout_rate': 0.7739280807288145, 'n_estimators': 132, 'learning_rate': 0.010648288131411126}. Best is trial 95 with value: 0.2259993215506233.
Fold 1 IBS: 0.24385115968166024
Fold 2 IBS: 0.21638959078128364
Fold 3 IBS: 0.2310974483968285
Fold 4 IBS: 0.23167317839356094
Fold 5 IBS: 0.21518657563145072
[I 2024-04-14 19:29:51,719] Trial 97 finished with value: 0.22763959057695682 and parameters: {'subsample

In [66]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.667
train_ibs:  0.224


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.3060837787360696,
                                              learning_rate=0.04133902891970489,
                                              n_estimators=1, random_state=123,
                                              subsample=0.1001314126580015)

C-index score: 0.535


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7782160113329367,
                                              learning_rate=0.005308151321564225,
                                              n_estimators=212,
                                              random_state=123,
                                              subsample=0.17778434918103095)

IBS: 0.233


In [70]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [71]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.830,1.0
ExtraSurvivalTrees,0.764,2.0
GradientBoosting,0.713,3.0
CoxLasso,0.691,4.5
CoxElastic,0.691,4.5
CoxPH,0.689,6.0
ComponentwiseGradientBoosting,0.667,7.0
CoxRidge,0.660,8.0


In [72]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.202,1.0
CoxPH,0.207,3.5
CoxLasso,0.207,3.5
CoxElastic,0.207,3.5
ExtraSurvivalTrees,0.207,3.5
ComponentwiseGradientBoosting,0.224,6.0
GradientBoosting,0.233,7.0
CoxRidge,0.236,8.0


In [73]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.544,1.0
ExtraSurvivalTrees,0.543,2.0
CoxRidge,0.535,3.5
ComponentwiseGradientBoosting,0.535,3.5
CoxPH,0.532,6.5
CoxLasso,0.532,6.5
CoxElastic,0.532,6.5
GradientBoosting,0.532,6.5


In [74]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
CoxRidge,0.229,1.5
GradientBoosting,0.229,1.5
ComponentwiseGradientBoosting,0.233,3.0
Randomsurvivalforest,0.247,4.0
ExtraSurvivalTrees,0.256,5.0
CoxLasso,0.288,6.5
CoxElastic,0.288,6.5
CoxPH,0.290,8.0


In [77]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/dfs/robust/rent/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_dfs_robust_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [78]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
